In [1]:
# !pip install -q -U "torchao>=0.16.0" sentence-transformers datasets accelerate peft trl transformers bitsandbytes

In [2]:
from datasets import load_dataset

print("Loading dataset...")
raw_ds = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1",
                      split="train[:5000]")

sample_ds = raw_ds.shuffle(seed=42).select(range(min(3_000, len(raw_ds))))

print(f"Sampled {len(sample_ds)} random examples.")

Loading dataset...
Sampled 3000 random examples.


In [3]:
import torch
import warnings
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    GenerationConfig
)
import random
import numpy as np

warnings.filterwarnings("ignore")

qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {llm_id}...")
tokenizer = AutoTokenizer.from_pretrained(llm_id, clean_up_tokenization_spaces=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id, quantization_config=qwen_bnb, device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
)

pipe.generation_config = GenerationConfig(
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [4]:
def generate_synthetic_pair(example, chunk_length=1500):

    doc = str(example.get("text", "")).strip()
    if len(doc) > chunk_length:
        random_start = random.randint(0, len(doc) - chunk_length)
        start_pos = doc.find(". ", random_start)
        start_idx = start_pos + 2 if start_pos != -1 else random_start
        end_pos = doc.find(".", start_idx + chunk_length)
        end_idx = end_pos + 1 if end_pos != -1 else len(doc)
        chunk = doc[start_idx:end_idx].strip()
    else:
        chunk = doc

    prompt = f"<|im_start|>user\nRead the following document chunk and generate a single, short, specific question that is directly answered by it.\n\nDocument: {chunk}\n\nQuestion:<|im_end|>\n<|im_start|>assistant\n"

    outputs = pipe(
        prompt,
        truncation=True,
        return_full_text=False,
    )
    question = outputs[0]["generated_text"].strip()

    return {"anchor": question, "positive": chunk}

In [5]:
print("Generating synthetic pairs (this may take a moment)...")
pair_ds = sample_ds.map(generate_synthetic_pair, remove_columns=sample_ds.column_names)
print("Generation Complete! Here are 3 examples:")
for i in range(min(3, len(pair_ds))):
    print(f"\n--- Example {i + 1} ---")
    print("Q:", pair_ds[i]["anchor"])
    print("A:", pair_ds[i]["positive"][:200], "...")

split_ds = pair_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

Generating synthetic pairs (this may take a moment)...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generation Complete! Here are 3 examples:

--- Example 1 ---
Q: What are some reasons why NVIDIA Quadro RTX GPUs and Dell Precision workstations are providing better performance?
A: The recent release of Nuke 12.1, the NukeX Cara VR toolset for working with 360-degree video, as well as Nuke s SphericalTransform and Bilateral nodes, takes advantage of new GPU-caching functionality ...

--- Example 2 ---
Q: What does the IBM Edge Application Manager provide?
A: Data scientists and DevOps teams can rely on solid support for the IBM Edge Application Manager no matter where on the network s edge their data is created and processed.
Running jobs on their own edg ...

--- Example 3 ---
Q: What are the key features of the proposed protocols for selective packet ordering in out-of-order networks?
A: We describe light-weight protocols for selective packet ordering in out-of-order networks that carry memory traffic. The protocols are designed for heterogeneous high-performance systems, in particu

In [6]:
from transformers import AutoConfig
from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType

embed_id = "nvidia/Nemotron-3-Embed-1B-BF16"

print(f"Loading base embedding model {embed_id}...")
embed_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16}, 
    )

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,)

peft_model = get_peft_model(embed_model[0].auto_model, lora_config)
embed_model[0].auto_model = peft_model
print("\nLoRA trainable parameters:")
peft_model.print_trainable_parameters()

Loading base embedding model nvidia/Nemotron-3-Embed-1B-BF16...


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}



LoRA trainable parameters:
trainable params: 2,097,152 || all params: 1,143,015,424 || trainable%: 0.1835


In [7]:
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,)
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,)
from sentence_transformers.sentence_transformer.trainer import (
    SentenceTransformerTrainer,)

loss = MultipleNegativesRankingLoss(embed_model)

args = SentenceTransformerTrainingArguments(
    output_dir="./nemotron-embed-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",)

trainer = SentenceTransformerTrainer(
    model=embed_model, args=args, train_dataset=train_ds, loss=loss)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,0.178740
100,0.060618
150,0.039419
200,0.012975
250,0.033340
300,0.009683
350,0.011530
400,0.044758
450,0.057876
500,0.010169


TrainOutput(global_step=1350, training_loss=0.02734737096009431, metrics={'train_runtime': 223.5413, 'train_samples_per_second': 12.078, 'train_steps_per_second': 6.039, 'total_flos': 0.0, 'train_loss': 0.02734737096009431, 'epoch': 1.0})

In [8]:
eval_queries = eval_ds["anchor"]
eval_docs = eval_ds["positive"]

def evaluate_retrieval(model, model_name):
    query_embs = model.encode(eval_queries)
    doc_embs = model.encode(eval_docs)
    similarities = np.dot(query_embs, doc_embs.T)
    top_doc_indices = np.argmax(similarities, axis=1)
    correct_indices = np.arange(len(eval_queries))
    correct_retrievals = np.sum(top_doc_indices == correct_indices)
    recall_at_1 = correct_retrievals / len(eval_queries)
    print(f"{model_name} Strict Recall@1: {recall_at_1 * 100:.2f}%")
    return recall_at_1

print("\n--- Evaluation Results ---")

original_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16})
finetuned_model = embed_model

original_score = evaluate_retrieval(original_model, "Original Model")
finetuned_score = evaluate_retrieval(finetuned_model, "Fine-Tuned Model")

improvement = finetuned_score - original_score
print(f"Absolute Improvement: +{improvement * 100:.2f}%")


--- Evaluation Results ---


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Original Model Strict Recall@1: 45.67%
Fine-Tuned Model Strict Recall@1: 81.00%
Absolute Improvement: +35.33%
